# XGBoost OOF Evaluation — Diagnostics and Interpretation


## 1. Imports and Configuration


In [ ]:
# Core libraries
import warnings
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.inspection import PartialDependenceDisplay

import xgboost as xgb
import shap

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

SAVE_DIR = Path('../../results/interpretable_model/xgboost')
PLOTS_DIR = SAVE_DIR / 'plots'
TABLES_DIR = SAVE_DIR / 'tables'

print(f'SAVE_DIR:  {SAVE_DIR.resolve()}')


## 2. Load Exported Artifacts


In [ ]:
# Load metadata generated in xgboost.ipynb
meta_candidates = list(TABLES_DIR.glob('oof_eval_meta_*.json'))
if not meta_candidates:
    raise FileNotFoundError('No oof_eval_meta_*.json file found in tables dir.')
meta_path = max(meta_candidates, key=lambda p: p.stat().st_mtime)

meta = json.loads(meta_path.read_text())
target_col = meta['target_col']
feature_cols = meta['feature_cols']
POOR_WELL_QUANTILE = meta.get('poor_well_quantile', 0.20)

model_data_path = Path(meta['model_data_path'])
model_path = Path(meta['model_path'])

model_df_oof = pd.read_csv(model_data_path)

X = model_df_oof[feature_cols]
y_oof_orig = model_df_oof['target_orig'].to_numpy()
y_oof_pred_orig = model_df_oof['oof_pred_orig'].to_numpy()

model = xgb.XGBRegressor()
model.load_model(model_path)

print(f'Loaded meta: {meta_path}')
print(f'Loaded model: {model_path}')
print(f'Loaded model data: {model_data_path}')
print(f'Target: {target_col} | Features: {len(feature_cols)}')


## 3. Model-Fit Diagnostics (OOF)


In [ ]:
residuals_oof = y_oof_orig - y_oof_pred_orig

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1) Predicted vs actual
axes[0, 0].scatter(y_oof_orig, y_oof_pred_orig, alpha=0.35, s=18, color='steelblue')
lims = [
    min(np.min(y_oof_orig), np.min(y_oof_pred_orig)),
    max(np.max(y_oof_orig), np.max(y_oof_pred_orig)),
]
axes[0, 0].plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
axes[0, 0].set_xlabel(f'Actual {target_col} (original scale)')
axes[0, 0].set_ylabel(f'Predicted {target_col} (original scale)')
axes[0, 0].set_title('Predicted vs Actual (OOF)')
axes[0, 0].legend()

# 2) Residuals vs predicted
axes[0, 1].scatter(y_oof_pred_orig, residuals_oof, alpha=0.35, s=18, color='steelblue')
axes[0, 1].axhline(0.0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Predicted (original scale)')
axes[0, 1].set_ylabel('Residuals (actual - predicted)')
axes[0, 1].set_title('Residuals vs Predicted (OOF)')

# 3) Residual histogram
axes[1, 0].hist(residuals_oof, bins=40, density=True, alpha=0.75, color='steelblue', edgecolor='black')
axes[1, 0].set_xlabel('Residual')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title(f'Residual Distribution (skew={stats.skew(residuals_oof):.3f})')

# 4) Q-Q plot
stats.probplot(residuals_oof, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot of Residuals')

plt.suptitle(f'XGBoost Residual Diagnostics — Target: {target_col}', fontsize=15, y=1.02)
plt.tight_layout()
diag_path = PLOTS_DIR / f'residual_diagnostics_oof_{target_col}.png'
plt.savefig(diag_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Diagnostics plot saved to: {diag_path}')
print(f'Residual mean:   {residuals_oof.mean():.6f}')
print(f'Residual median: {np.median(residuals_oof):.6f}')
print(f'Residual std:    {residuals_oof.std():.6f}')


## 4. SHAP Interpretability (Global + Local)


In [ ]:
# Build SHAP explanation on full data (final model)
explainer = shap.Explainer(model, X)
shap_exp = explainer(X)
shap_values = shap_exp.values

# Global importance (mean absolute SHAP)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance_df = pd.DataFrame({
    'feature': X.columns,
    'mean_abs_shap': mean_abs_shap,
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

display(shap_importance_df)

shap_imp_path = TABLES_DIR / f'shap_importance_{target_col}.csv'
shap_importance_df.to_csv(shap_imp_path, index=False)
print(f'SHAP importance table saved to: {shap_imp_path}')

# Keep top features for downstream PDP section
top_features = shap_importance_df['feature'].head(6).tolist()
print('Top 6 features by mean |SHAP|:')
print(top_features)

# SHAP beeswarm
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X, show=False, max_display=20)
beeswarm_path = PLOTS_DIR / f'shap_beeswarm_{target_col}.png'
plt.tight_layout()
plt.savefig(beeswarm_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close()

# SHAP bar
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X, plot_type='bar', show=False, max_display=20)
bar_path = PLOTS_DIR / f'shap_bar_{target_col}.png'
plt.tight_layout()
plt.savefig(bar_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close()

print(f'SHAP beeswarm saved to: {beeswarm_path}')
print(f'SHAP bar saved to:      {bar_path}')


## 5. Partial Dependence Plots (Top 6 SHAP Features)


In [ ]:
for feat in top_features:
    fig, ax = plt.subplots(figsize=(8, 5))
    PartialDependenceDisplay.from_estimator(
        model,
        X,
        features=[feat],
        kind='average',
        grid_resolution=40,
        ax=ax,
    )
    ax.set_title(f'PDP — {feat}')
    pdp_path = PLOTS_DIR / f'pdp_{feat}_{target_col}.png'
    plt.tight_layout()
    plt.savefig(pdp_path, dpi=150, bbox_inches='tight')
    plt.show()

print('PDP plots saved for top 6 SHAP-ranked features.')


## 6. Cohort Analysis: Poor vs Well Performance (OOF)


In [ ]:
abs_error_oof = np.abs(y_oof_orig - y_oof_pred_orig)
poor_threshold = np.quantile(abs_error_oof, 1.0 - POOR_WELL_QUANTILE)
well_threshold = np.quantile(abs_error_oof, POOR_WELL_QUANTILE)

poor_mask = abs_error_oof >= poor_threshold
well_mask = abs_error_oof <= well_threshold

X_reset = X.reset_index(drop=True).copy()
X_reset['abs_error'] = abs_error_oof

poor_df = X_reset.loc[poor_mask].copy()
well_df = X_reset.loc[well_mask].copy()

print(f'Poor threshold (top {POOR_WELL_QUANTILE:.0%}): abs_error >= {poor_threshold:.6f}')
print(f'Well threshold (bottom {POOR_WELL_QUANTILE:.0%}): abs_error <= {well_threshold:.6f}')
print(f'Poor cohort size: {len(poor_df)}')
print(f'Well cohort size: {len(well_df)}')


In [ ]:
def standardized_difference(a, b, eps=1e-12):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    pooled = np.sqrt((np.var(a, ddof=1) + np.var(b, ddof=1)) / 2.0)
    if not np.isfinite(pooled) or pooled < eps:
        return 0.0
    return (np.mean(a) - np.mean(b)) / pooled

cohort_rows = []
for feat in X.columns:
    poor_vals = poor_df[feat].dropna().values
    well_vals = well_df[feat].dropna().values

    cohort_rows.append({
        'feature': feat,
        'poor_mean': np.mean(poor_vals),
        'well_mean': np.mean(well_vals),
        'poor_median': np.median(poor_vals),
        'well_median': np.median(well_vals),
        'mean_diff': np.mean(poor_vals) - np.mean(well_vals),
        'median_diff': np.median(poor_vals) - np.median(well_vals),
        'standardized_diff': standardized_difference(poor_vals, well_vals),
    })

cohort_df = pd.DataFrame(cohort_rows)
cohort_df['abs_standardized_diff'] = cohort_df['standardized_diff'].abs()
cohort_df = cohort_df.sort_values('abs_standardized_diff', ascending=False).reset_index(drop=True)

display(cohort_df)

cohort_path = TABLES_DIR / f'cohort_comparison_{target_col}.csv'
cohort_df.to_csv(cohort_path, index=False)
print(f'Cohort comparison table saved to: {cohort_path}')


In [ ]:
# Violin plots for top differentiating features
top_diff_features = cohort_df['feature'].head(6).tolist()
print('Top differentiating features (|standardized_diff|):')
print(top_diff_features)

n_plot = len(top_diff_features)
n_cols = 3
n_rows = int(np.ceil(n_plot / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for i, feat in enumerate(top_diff_features):
    long_df = pd.concat([
        pd.DataFrame({'cohort': 'Well (bottom 20%)', 'value': well_df[feat].values}),
        pd.DataFrame({'cohort': 'Poor (top 20%)', 'value': poor_df[feat].values}),
    ], ignore_index=True)

    sns.violinplot(data=long_df, x='cohort', y='value', ax=axes[i], inner='box', cut=0)
    axes[i].set_title(feat)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=20)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Poor vs Well Cohorts — Feature Distributions', fontsize=15, y=1.02)
plt.tight_layout()
cohort_plot_path = PLOTS_DIR / f'cohort_violin_top_features_{target_col}.png'
plt.savefig(cohort_plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Cohort violin plots saved to: {cohort_plot_path}')


## 7. Sanity Checks and Final Summary


In [ ]:
# Sanity check 1: poor median error > well median error
poor_median_error = np.median(abs_error_oof[poor_mask])
well_median_error = np.median(abs_error_oof[well_mask])
print(f'Poor median abs error: {poor_median_error:.6f}')
print(f'Well median abs error: {well_median_error:.6f}')
print(f'Check poor > well: {poor_median_error > well_median_error}')

# Sanity check 2: overlap between top SHAP and top cohort-difference features
top_shap = set(shap_importance_df['feature'].head(6).tolist())
top_cohort = set(cohort_df['feature'].head(6).tolist())
overlap = top_shap.intersection(top_cohort)

print(f'Top SHAP features:   {sorted(top_shap)}')
print(f'Top cohort features: {sorted(top_cohort)}')
print(f'Overlap ({len(overlap)}): {sorted(overlap)}')

print('\nSaved artifacts:')
print(f'- SHAP importance table:  {shap_imp_path}')
print(f'- Cohort comparison:      {cohort_path}')
print(f'- Plot directory:         {PLOTS_DIR}')
